# EverFlavor AI - Cooking Yields, Nutrient Retention and Safe Temperatures

**Team:** EverGlow | **Course:** ITAI 2277 | **Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: new (2026-11-19), sections 2 to 4 done, section 5 waiting for its source.** Runs in under a minute;
> downloads about 0.5 MB once. Needs nothing else.

---

## Purpose

Cooking changes food: meat loses water and fat, and some vitamins are lost in the pot. This notebook turns two
USDA tables into tables the app and the waiter can read, so a recipe's weight and nutrition can be given for the
cooked food, and the waiter can answer "why did my chicken shrink?" from data.

| Section | What it does | Output |
|---|---|---|
| 2 | Cooking yields: raw to cooked weight, water and fat change, for 173 cuts of meat and poultry and their methods | `data/processed/cooking_yields.json` |
| 3 | Nutrient retention: how much of 26 vitamins, minerals and alcohol is kept, for 270 foods and preparations | `data/processed/nutrient_retention.json` |
| 4 | Cooking method profiles: each method's yield range and typical vitamin losses, from sections 2 and 3 | `data/processed/cooking_method_profiles.json` |
| 5 | Safe minimum internal temperatures (USDA FSIS): **waiting for the source** | `data/processed/safe_temperatures.json` (later) |

**Rules:** every number comes from the USDA tables; nothing is estimated or filled in. A value the table does not
have stays empty. The words that name each method (`method_key`) are matched by fixed rules listed in the cells,
so the team can check them.

**Sources** (both US Department of Agriculture, Agricultural Research Service; CC0 1.0 on data.gov, checked
2026-11-19):

- USDA Table of Cooking Yields for Meat and Poultry, Release 2 (2014). DOI 10.15482/USDA.ADC/1409031.
- USDA Table of Nutrient Retention Factors, Release 6 (2007). DOI 10.15482/USDA.ADC/1409034.

---

## 1. Setup

The same setup as the other notebooks: it works unchanged in Google Colab, VS Code and Antigravity.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook
%pip install -q pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "agent_tools", "calories", "charts", "checks", "confidence", "cooking", "cooking_time", "dish_steps", "crew",
                      "cuisine", "diets", "dish_knowledge", "dish_names", "environment", "evaluation", "features", "flags", "freshness", "ingredients",
                      "knowledge", "llm", "nutrition", "nutrition_quality", "parsing", "pipeline", "places", "preferences", "preparation", "recipe_details",
                      "progress", "recommend", "refresh", "reporting", "review", "safety", "seasons", "sources",
                      "saved_recipes", "stores", "variants", "waiter", "recipe_check", "restrictions", "taxonomy"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

OUT = Path("data/processed")
print(f"Running from {short_path(PROJECT_ROOT)} ({'Colab' if IN_COLAB else 'local'})")

Running from ~\Downloads\jupiter-exploration\The-EverFlavor-AI (local)


In [3]:
import json
import re
import urllib.request

RAW = Path("data/raw/usda_cooking")
FILES = {   # data.gov's download links (figshare, the USDA Ag Data Commons store)
    "CookingYields.csv": "https://ndownloader.figshare.com/files/43365402",
    "CookingYieldsDD.csv": "https://ndownloader.figshare.com/files/43365405",
    "NutrientRetention.csv": "https://ndownloader.figshare.com/files/44488754",
    "NutrientRetentionDD.csv": "https://ndownloader.figshare.com/files/44488757",
}
RAW.mkdir(parents=True, exist_ok=True)
for name, url in FILES.items():
    if not (RAW / name).exists():
        partial = RAW / f"{name}.part"
        urllib.request.urlretrieve(url, partial)
        partial.replace(RAW / name)
        print(f"Downloaded {name}")
print("Files:", ", ".join(f"{n} ({(RAW / n).stat().st_size / 1024:.0f} KB)" for n in FILES))

Files: CookingYields.csv (21 KB), CookingYieldsDD.csv (2 KB), NutrientRetention.csv (462 KB), NutrientRetentionDD.csv (1 KB)


---

## 2. Cooking yields

The cooked weight as a share of the raw weight (`yield_factor`: 0.69 means 200 g raw becomes 138 g cooked), with
the change in water and fat, the number of samples and their range. USDA's own method names are kept
(`usda_method`); `method_key` puts them in the project's cooking-method names (`cooking.COOKING_METHODS`), plus
`braise`, which that list does not have yet.

In [4]:
from everflavor.cooking import COOKING_METHODS

yields = pd.read_csv(RAW / "CookingYields.csv", encoding="latin-1")
yields.columns = [re.sub(r"\s+", " ", c).strip() for c in yields.columns]

# USDA's method names -> the project's method keys (a fixed list, for the team to check)
YIELD_METHODS = {
    "baked or roasted, unspecified": "bake_roast", "baked or roasted, specified time": "bake_roast",
    "roasted": "bake_roast", "broiled or grilled": "grill_broil", "grilled": "grill_broil", "broiled": "grill_broil",
    "braised": "braise", "poached, simmered or stewed": "boil_simmer", "pan-fried": "pan_fry",
    "pan-fried or sauteed": "pan_fry", "pan-broiled": "pan_fry", "pan broiled": "pan_fry", "pan-browned": "pan_fry",
    "fried in deep fat": "deep_fry", "microwaved": "microwave", "multiple cooking methods": "multiple",
}


def percent(value):
    """A USDA percent as a share (69 -> 0.69); None when the table has no value."""
    return None if pd.isna(value) else round(float(value) / 100, 4)


rows = []
for r in yields.to_dict("records"):
    usda_method = re.sub(r"\s+", " ", str(r["Preparation Method1"])).strip()
    description = str(r["Yield Description"]).strip()
    rows.append({
        "food": description, "meat": description.split(",")[0].strip().lower(),
        "ndb": None if pd.isna(r["NDB"]) else int(r["NDB"]), "usda_method": usda_method,
        "method_key": YIELD_METHODS.get(usda_method.lower(), "other"),
        "yield_factor": percent(r["Cooking Yield %"]), "yield_min": percent(r["Yield Minimum %"]),
        "yield_max": percent(r["Yield Maximum %"]), "moisture_change": percent(r["Moisture Gain/ Loss %"]),
        "fat_change": percent(r["Fat Gain/ Loss %"]), "samples": None if pd.isna(r["n"]) else int(r["n"]),
        "release_year": int(r["Release Year2"]),
    })
cooking_yields = pd.DataFrame(rows)
unknown_keys = set(cooking_yields["method_key"]) - {"braise", "multiple", "other"} - set(COOKING_METHODS)
if unknown_keys:
    raise ValueError(f"method keys the project does not know: {unknown_keys}")
print(f"{len(cooking_yields)} yields; methods not matched: {(cooking_yields['method_key'] == 'other').sum()}")
print(cooking_yields.groupby("method_key")["yield_factor"].describe()[["count", "min", "50%", "max"]].round(2))

173 yields; methods not matched: 0
             count   min   50%   max
method_key                          
bake_roast    55.0  0.32  0.77  0.96
boil_simmer   19.0  0.36  0.70  0.86
braise        28.0  0.60  0.71  0.79
deep_fry       1.0  0.66  0.66  0.66
grill_broil   45.0  0.63  0.78  0.86
microwave      1.0  0.29  0.29  0.29
multiple       1.0  0.76  0.76  0.76
pan_fry       23.0  0.32  0.73  0.86


### 2.1 An example: chicken breast

What 200 g of raw chicken breast (meat and skin) weighs when it is done, by the method USDA measured.

In [5]:
breast = cooking_yields[cooking_yields["food"].str.contains("Chicken, broiler-fryer, breast", case=False)]
def change(value):
    return "not measured" if pd.isna(value) else f"{value:+.0%}"


for r in breast.itertuples():
    print(f"{r.usda_method}: 200 g raw -> {200 * r.yield_factor:.0f} g cooked (yield {r.yield_factor:.0%}; "
          f"water {change(r.moisture_change)}, fat {change(r.fat_change)}; USDA release {r.release_year})")
print("(USDA's 1975 rows have the yield only, not the water and fat change.)")

Baked or Roasted, unspecified: 200 g raw -> 144 g cooked (yield 72%; water not measured, fat not measured; USDA release 1975)
Poached, Simmered or Stewed: 200 g raw -> 154 g cooked (yield 77%; water not measured, fat not measured; USDA release 1975)
(USDA's 1975 rows have the yield only, not the water and fat change.)


### 2.2 By meat and method

The middle (median) yield for each meat and method, with how many cuts USDA measured.

In [6]:
by_meat = (cooking_yields.groupby(["meat", "method_key"])["yield_factor"]
           .agg(cuts="count", median="median", low="min", high="max").round(2).reset_index())
print(by_meat[by_meat["cuts"] >= 2].to_string(index=False))

     meat  method_key  cuts  median  low  high
     beef  bake_roast    13    0.79 0.63  0.85
     beef boil_simmer     4    0.64 0.53  0.85
     beef      braise    14    0.68 0.64  0.75
     beef grill_broil    27    0.77 0.63  0.86
     beef     pan_fry     7    0.69 0.62  0.77
  chicken  bake_roast     8    0.74 0.61  0.80
  chicken boil_simmer     7    0.74 0.55  0.86
  chicken      braise     4    0.73 0.68  0.77
game meat grill_broil     3    0.83 0.77  0.84
     pork  bake_roast    22    0.84 0.32  0.96
     pork boil_simmer     3    0.39 0.36  0.69
     pork      braise     8    0.75 0.73  0.79
     pork grill_broil    10    0.80 0.68  0.85
     pork     pan_fry    13    0.77 0.32  0.86
   turkey  bake_roast    10    0.72 0.68  0.83
   turkey boil_simmer     4    0.80 0.64  0.85
     veal grill_broil     2    0.76 0.74  0.77


In [7]:
OUT.mkdir(parents=True, exist_ok=True)
SOURCE_YIELDS = {"name": "USDA Table of Cooking Yields for Meat and Poultry, Release 2 (2014)",
                 "publisher": "USDA Agricultural Research Service", "licence": "CC0 1.0",
                 "doi": "10.15482/USDA.ADC/1409031", "url": FILES["CookingYields.csv"], "retrieved": "2026-11-19"}
records = json.loads(cooking_yields.to_json(orient="records"))   # NaN -> null
(OUT / "cooking_yields.json").write_text(json.dumps({"source": SOURCE_YIELDS, "yields": records}, indent=1),
                                         encoding="utf-8")
print(f"Wrote {len(records)} yields to {OUT / 'cooking_yields.json'}")

Wrote 173 yields to data\processed\cooking_yields.json


---

## 3. Nutrient retention

For each food and preparation, the share of each nutrient that is still there after cooking (`1.0`: all of it).
USDA writes the preparation in short words ("LEGUMES,CKD 2/2.5HRS,BLD,DRND,FRIED"); the methods are read from
those words by the rules below. A preparation with two methods (boiled, then fried) is `multiple`; reheating is
`reheat`; a row with none of the words (canned, fresh) is `none`.

The food groups are USDA's Standard Reference groups (the codes in the table).

In [8]:
retention = pd.read_csv(RAW / "NutrientRetention.csv", encoding="latin-1")
# A few rows have a date where the factor should be: they have no factor and are left out (not guessed)
retention["Retn_Factor"] = pd.to_numeric(retention["Retn_Factor"], errors="coerce")
no_factor = retention[retention["Retn_Factor"].isna()]
print(f"Rows without a factor, left out: {len(no_factor)} ({', '.join(sorted(set(no_factor['RetnDesc'])))})")
retention = retention.dropna(subset=["Retn_Factor"])
FOOD_GROUPS = {1: "Dairy and Egg Products", 5: "Poultry Products", 8: "Breakfast Cereals",
               9: "Fruits and Fruit Juices", 10: "Pork Products", 11: "Vegetables and Vegetable Products",
               12: "Nut and Seed Products", 13: "Beef Products", 14: "Beverages", 15: "Finfish and Shellfish Products",
               16: "Legumes and Legume Products", 17: "Lamb, Veal, and Game Products",
               20: "Cereal Grains and Pasta"}
RETENTION_METHODS = [   # (method key, the USDA words that mean it), checked in this order
    ("deep_fry", r"DEEP[ -]?F"), ("stir_fry", r"STIR[ -]?FR"),
    ("pan_fry", r"\bFRIED\b|\bFRY\b|SAUTE"), ("bake_roast", r"\bBAKED\b|\bBKD\b|ROAST"),
    ("grill_broil", r"BROIL|GRILL"), ("braise", r"BRAISE"), ("steam", r"STEAM"),
    ("boil_simmer", r"\bBOIL|\bBLD\b|SIMM|STEW|POACH|WATER USED|\bCKD\b|COOKED"),
    ("microwave", r"MICROWAV"), ("reheat", r"REHEAT|HEATED"),
]


def methods_in(description):
    words = description.upper()
    return [key for key, pattern in RETENTION_METHODS if re.search(pattern, words)]


foods = retention.drop_duplicates("Retn_Code")[["Retn_Code", "FdGrp_CD", "RetnDesc"]]
profiles = []
for f in foods.itertuples():
    found = methods_in(f.RetnDesc)
    cooked = [m for m in found if m != "reheat"]
    key = "multiple" if len(cooked) > 1 else cooked[0] if cooked else "reheat" if found else "none"
    factors = retention[retention["Retn_Code"] == f.Retn_Code]
    profiles.append({"retention_code": int(f.Retn_Code), "food_group": FOOD_GROUPS.get(int(f.FdGrp_CD), ""),
                     "usda_description": f.RetnDesc.strip(), "method_key": key, "methods": cooked,
                     "retention": {str(n).strip(): round(v / 100, 2) for n, v in
                                   zip(factors["NutrDesc"], factors["Retn_Factor"])}})
retention_table = pd.DataFrame(profiles)
print(f"{len(retention_table)} foods and preparations, {retention['NutrDesc'].nunique()} nutrients")
print(retention_table["method_key"].value_counts().to_string())
print(retention_table["food_group"].value_counts().to_string())

Rows without a factor, left out: 24 (ALC BEV,STIRRED,BKD/SIMMRD 30 MIN)


270 foods and preparations, 26 nutrients


method_key
boil_simmer    99
multiple       40
reheat         32
bake_roast     31
pan_fry        29
grill_broil    19
none           16
steam           4
food_group
Vegetables and Vegetable Products    53
Finfish and Shellfish Products       40
Pork Products                        38
Beef Products                        26
Lamb, Veal, and Game Products        23
Legumes and Legume Products          19
Cereal Grains and Pasta              14
Dairy and Egg Products               13
Poultry Products                     13
Fruits and Fruit Juices              13
Beverages                            10
Breakfast Cereals                     4
Nut and Seed Products                 4


### 3.1 What cooking does to vitamin C, thiamin and folate

Medians across the vegetables, by method. A value of 0.75 means a quarter is lost.

In [9]:
KEY_NUTRIENTS = ["Vitamin C, total ascorbic acid", "Thiamin", "Folate, total", "Potassium, K"]
veg = retention_table[retention_table["food_group"] == "Vegetables and Vegetable Products"]
table = pd.DataFrame([{"method_key": r.method_key, **{n: r.retention.get(n) for n in KEY_NUTRIENTS}}
                      for r in veg.itertuples()])
print(table.groupby("method_key").agg(["median", "count"]).round(2).to_string())

            Vitamin C, total ascorbic acid       Thiamin       Folate, total       Potassium, K      
                                    median count  median count        median count       median count
method_key                                                                                           
bake_roast                            0.80     7    0.85     7          0.80     7          1.0     7
boil_simmer                           0.75    23    0.80    23          0.70    23          0.9    23
grill_broil                           0.85     1    0.90     1          0.95     1          1.0     1
multiple                              0.82     8    0.90     8          0.80     8          1.0     8
none                                  0.75     1    0.80     1          0.75     1          0.9     1
pan_fry                               0.80     5    0.85     5          0.75     5          1.0     5
reheat                                0.95     6    0.95     6          0.95     6

In [10]:
SOURCE_RETENTION = {"name": "USDA Table of Nutrient Retention Factors, Release 6 (2007)",
                    "publisher": "USDA Agricultural Research Service", "licence": "CC0 1.0",
                    "doi": "10.15482/USDA.ADC/1409034", "url": FILES["NutrientRetention.csv"],
                    "retrieved": "2026-11-19"}
(OUT / "nutrient_retention.json").write_text(
    json.dumps({"source": SOURCE_RETENTION, "foods": profiles}, indent=1), encoding="utf-8")
print(f"Wrote {len(profiles)} retention profiles to {OUT / 'nutrient_retention.json'}")

Wrote 270 retention profiles to data\processed\nutrient_retention.json


---

## 4. Cooking method profiles

One entry per method, from sections 2 and 3 only: the range of meat yields USDA measured, and the median share of
each key nutrient kept across all foods prepared that way. Notebook 16 (techniques and cookware) will add what each
method is, from its own sources; nothing here is a description written by us.

In [11]:
method_profiles = {}
for key in sorted(set(cooking_yields["method_key"]) | set(retention_table["method_key"])):
    if key in {"other", "none"}:
        continue
    meat = cooking_yields[cooking_yields["method_key"] == key]["yield_factor"].dropna()
    rows = retention_table[retention_table["method_key"] == key]
    kept = {n: rows["retention"].map(lambda d, n=n: d.get(n)).dropna() for n in KEY_NUTRIENTS}
    method_profiles[key] = {
        "meat_yield": None if meat.empty else {"low": round(meat.min(), 2), "median": round(meat.median(), 2),
                                                "high": round(meat.max(), 2), "cuts": int(meat.size)},
        "nutrients_kept_median": {n: round(float(v.median()), 2) for n, v in kept.items() if len(v)},
        "retention_foods": len(rows),
    }
print(pd.DataFrame({k: {"meat yield": (v["meat_yield"] or {}).get("median"), "cuts": (v["meat_yield"] or {}).get("cuts"),
                        "vitamin C kept": v["nutrients_kept_median"].get(KEY_NUTRIENTS[0]),
                        "foods": v["retention_foods"]} for k, v in method_profiles.items()}).T.to_string())
(OUT / "cooking_method_profiles.json").write_text(json.dumps(
    {"sources": [SOURCE_YIELDS, SOURCE_RETENTION], "methods": method_profiles}, indent=1), encoding="utf-8")
print(f"Wrote {len(method_profiles)} method profiles")

             meat yield  cuts  vitamin C kept  foods
bake_roast         0.77  55.0            0.80   31.0
boil_simmer        0.70  19.0            0.80   99.0
braise             0.71  28.0             NaN    0.0
deep_fry           0.66   1.0             NaN    0.0
grill_broil        0.78  45.0            0.80   19.0
microwave          0.29   1.0             NaN    0.0
multiple           0.76   1.0            0.80   40.0
pan_fry            0.73  23.0            0.80   29.0
reheat              NaN   NaN            0.95   32.0
steam               NaN   NaN            0.80    4.0
Wrote 10 method profiles


---

## 5. Safe minimum internal temperatures (waiting for the source)

The USDA FSIS chart (<https://www.fsis.usda.gov/food-safety/safe-food-handling-and-preparation/food-safety-basics/safe-temperature-chart>)
is the source. The site refuses automated downloads, so the page is saved by hand from a browser into
`data/raw/usda_cooking/` (with the date it was saved); this section then reads its table into
`data/processed/safe_temperatures.json`, with the source, the date and FSIS as the credit. Until then it only
says whether the file is there.

In [12]:
saved = sorted(RAW.glob("*safe*temperature*.htm*")) + sorted(RAW.glob("*safe*temperature*.csv"))
print("Saved chart:", ", ".join(p.name for p in saved) if saved else "not yet (see the text above)")

Saved chart: not yet (see the text above)


---

## 6. The knowledge-asset list

`data/processed/knowledge_assets.json` lists the knowledge files that exist, so the app, the waiter and later
notebooks can see what is there, how big it is and where it comes from without opening each file. Each notebook
adds or replaces its own entries and keeps the others.

In [13]:
from datetime import date

MANIFEST = OUT / "knowledge_assets.json"
assets = json.loads(MANIFEST.read_text(encoding="utf-8")) if MANIFEST.exists() else {}
built = date.today().isoformat()
assets.update({
    "cooking_yields": {"file": "data/processed/cooking_yields.json", "records": len(records),
                       "unit": "cut of meat or poultry and cooking method", "source": SOURCE_YIELDS["name"],
                       "licence": "CC0 1.0", "notebook": "14", "built": built},
    "nutrient_retention": {"file": "data/processed/nutrient_retention.json", "records": len(profiles),
                           "factors": int(retention["Retn_Factor"].notna().sum()),
                           "unit": "food and preparation (26 nutrients each)", "source": SOURCE_RETENTION["name"],
                           "licence": "CC0 1.0", "notebook": "14", "built": built},
    "cooking_method_profiles": {"file": "data/processed/cooking_method_profiles.json",
                                "records": len(method_profiles), "unit": "cooking method",
                                "source": "derived", "derived_from": ["cooking_yields", "nutrient_retention"],
                                "notebook": "14", "built": built},
})
MANIFEST.write_text(json.dumps(dict(sorted(assets.items())), indent=1), encoding="utf-8")
for name, entry in sorted(assets.items()):
    print(f"{name}: {entry['records']} ({entry['unit']}); source: {entry['source']}")

cooking_method_profiles: 10 (cooking method); source: derived
cooking_yields: 173 (cut of meat or poultry and cooking method); source: USDA Table of Cooking Yields for Meat and Poultry, Release 2 (2014)
nutrient_retention: 270 (food and preparation (26 nutrients each)); source: USDA Table of Nutrient Retention Factors, Release 6 (2007)


---

## Summary

- Section 2: 173 USDA cooking yields, each in the project's method names, so a recipe's meat can be given as cooked
  weight; the chicken example shows the use.
- Section 3: 270 USDA retention profiles (26 nutrients each), for nutrition after cooking and for the waiter.
- Section 4: method profiles built only from those two tables, ready for notebook 16.
- Next: section 5 once the FSIS chart is saved; then the app's nutrition estimate can use sections 2 and 3 (a
  change in `src/everflavor`, planned separately).